In [33]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [34]:
load_dotenv()
model = ChatOpenAI(model='gpt-5.6-luna')

In [35]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    score: int

In [36]:
def create_outline(state: BlogState) -> BlogState:

    ##fetch title
    title = state['title']

    ##prompt
    prompt = f"Generate a detailed outline for a 300 word blog on the topic - {title}"
    outline = model.invoke(prompt).content

    ##update state
    state['outline'] = outline

    return state

In [37]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a 300 word blog on the topic {title}, use the following outline \n {outline}'

    blog = model.invoke(prompt).content

    state['content'] = blog

    return state


In [38]:
def eval_blog(state: BlogState) -> BlogState:

    outline = state['outline']
    blog = state['content']

    prompt = f'Based on the outline, rate the generated blog out of 10(give an integer rating). Outline: \n {outline}. Blog: \n {blog}'

    state['score'] = model.invoke(prompt).content

    return state

In [39]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', eval_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [40]:
initial_state = {'title': 'Evolution of the tech-job market in India'}
final_state = workflow.invoke(initial_state)

In [41]:
print(final_state)

{'title': 'Evolution of the tech-job market in India', 'outline': '# Detailed Outline: *Evolution of the Tech-Job Market in India*\n\n**Target length: 300 words**\n\n## 1. Introduction: From IT Services to a Digital Economy — 35 words\n- Introduce India’s transformation from a major IT-outsourcing destination into a global technology and innovation hub.\n- Mention how technology jobs have expanded beyond software services into startups, fintech, artificial intelligence, cybersecurity, cloud computing, and product development.\n- Central idea: India’s tech-job market has grown rapidly but is also becoming more competitive and skills-driven.\n\n## 2. The Early Phase: IT Services and Outsourcing — 55 words\n- Discuss the growth of companies such as TCS, Infosys, Wipro, and HCL.\n- Explain how liberalization, lower operating costs, a large English-speaking workforce, and global demand for software services supported expansion.\n- Note that early jobs mainly involved programming, business-p

In [42]:
print(final_state['outline'])

# Detailed Outline: *Evolution of the Tech-Job Market in India*

**Target length: 300 words**

## 1. Introduction: From IT Services to a Digital Economy — 35 words
- Introduce India’s transformation from a major IT-outsourcing destination into a global technology and innovation hub.
- Mention how technology jobs have expanded beyond software services into startups, fintech, artificial intelligence, cybersecurity, cloud computing, and product development.
- Central idea: India’s tech-job market has grown rapidly but is also becoming more competitive and skills-driven.

## 2. The Early Phase: IT Services and Outsourcing — 55 words
- Discuss the growth of companies such as TCS, Infosys, Wipro, and HCL.
- Explain how liberalization, lower operating costs, a large English-speaking workforce, and global demand for software services supported expansion.
- Note that early jobs mainly involved programming, business-process outsourcing, maintenance, testing, and technical support.
- Mention the 

In [43]:
print(final_state['content'])

# Evolution of the Tech-Job Market in India

India’s tech-job market has evolved from an outsourcing powerhouse to a digital economy and global innovation hub. Careers now span startups, fintech, artificial intelligence, cybersecurity, cloud computing, and product development. Growth is rapid, but competition is increasingly skills-driven.

In the early phase, TCS, Infosys, Wipro, and HCL drove India’s rise as a software-services destination. Economic liberalization, lower operating costs, a large English-speaking workforce, and global demand created scale. Jobs centered on programming, business-process outsourcing, maintenance, testing, and technical support. Bengaluru, Hyderabad, Pune, Chennai, and Gurugram grew into technology hubs, attracting graduates nationwide.

Affordable internet, smartphones, digital payments, and initiatives such as Digital India transformed employment. Startups expanded across e-commerce, fintech, edtech, healthtech, logistics, and SaaS, creating opportunit

In [44]:
print(final_state['score'])

**Rating: 9/10**

The blog closely follows the outline, covers all major themes, maintains a clear chronological structure, and includes the specified companies, technologies, roles, challenges, and future trends. It is well-written and coherent. The only minor issue is that it appears somewhat longer than the 300-word target.
